# Tshivenda Whisper Pilot Fine-Tune (M4 / MPS)

Wraps `src/asr/pilot_finetune_whisper_mps_ven.py` - the reduced overnight pilot
fine-tune run on Apple Silicon, same role as `pilot_finetune_mps.ipynb` but
for Whisper instead of Wav2Vec2 (the second of the two ASR model families,
see `notes/pilot-ven-results.md` for why AfriHuBERT was dropped as a third).
NOT the real Stage 1 run (whisper-large-v3, full data, GPU) - this is for
quick local iteration and sanity checks only.

Uses "sw" (Swahili) as a placeholder language token since Whisper has no
`<|ven|>` token - see the script's module docstring for the reasoning.

**Kernel**: MultilingualASR.

In [ ]:
import os
os.environ["DYLD_FALLBACK_LIBRARY_PATH"] = "/opt/homebrew/opt/ffmpeg@7/lib"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

## Config

In [ ]:
MODEL = "openai/whisper-small"   # swap to openai/whisper-large-v3 for reported numbers (slow locally)
TRAIN_CLIPS = 5000
EVAL_CLIPS = 500
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8
MAX_INPUT_SECONDS = 10.0   # memory cap - see notes/pilot-ven-results.md for why
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.1
PATIENCE = 3
FREEZE_ENCODER = False
RESUME_FROM = None          # e.g. "../results/whisper-ven-pilot/final" to continue
INCLUDE_ANV = False

## Run

Slow (this pilot took ~4.5h on an M4 for 3 epochs / 5,000 clips) - for a real
overnight run, launch `src/asr/pilot_finetune_whisper_mps_ven.py` from the terminal
in the background instead (see notes/pilot-ven-results.md for the exact
commands and heartbeat-monitoring pattern). This cell is for short local
test runs (e.g. `TRAIN_CLIPS = 50, EVAL_CLIPS = 20, EPOCHS = 1` as a smoke test).

In [ ]:
import sys, argparse
sys.path.insert(0, "../../src/asr")
import pilot_finetune_whisper_mps_ven as pft

args = argparse.Namespace(
    model=MODEL, train_clips=TRAIN_CLIPS, eval_clips=EVAL_CLIPS, epochs=EPOCHS,
    batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM, max_input_seconds=MAX_INPUT_SECONDS,
    learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO, patience=PATIENCE,
    freeze_encoder=FREEZE_ENCODER, resume_from=RESUME_FROM, include_anv=INCLUDE_ANV,
)
pft.main(args)

## Pilot v1 results (already run - see `results/logs/whisper_pilot_v1_wer0265.log`)

whisper-small, 5,000 NCHLT train clips (<= 10s, 4,974 kept), 493 eval clips,
3 epochs, M4 MacBook (MPS). Steady improvement every epoch, no collapse.

| Epoch | eval WER | eval CER |
|---|---|---|
| 1 | 0.428 | 0.110 |
| 2 | 0.281 | 0.064 |
| 3 (final) | **0.265** | **0.060** |

Comparison (200-clip NCHLT test set, seed 42):

| Model | WER | CER |
|---|---|---|
| Whisper Large v3 zero-shot | 1.108 | 0.763 |
| Wav2Vec2 pilot v2 (+ANV, 8 total epochs) | 0.332 | 0.074 |
| **Whisper pilot v1 (3 epochs, NCHLT only)** | **0.265** | **0.060** |

Best result of either model family so far, in fewer epochs and without the
extra ANV data Wav2Vec2 needed to get there. Model checkpoint saved to
`results/whisper-ven-pilot/final` (gitignored - re-run this notebook or the
script to regenerate).